In [1]:
import pandas as pd 
import numpy as np 
from sklearn.linear_model import LinearRegression 
df=pd.read_csv(r"C:\Users\Amin\Downloads\covid_19_data.csv") 
df.head()

,SNo,ObservationDate,Province/State,Country/Region,Last Update,Confirmed,Deaths,Recovered
0,1,01/22/2020,Anhui,Mainland China,1/22/2020 17:00,1.0,0.0,0.0
1,2,01/22/2020,Beijing,Mainland China,1/22/2020 17:00,14.0,0.0,0.0
2,3,01/22/2020,Chongqing,Mainland China,1/22/2020 17:00,6.0,0.0,0.0
3,4,01/22/2020,Fujian,Mainland China,1/22/2020 17:00,1.0,0.0,0.0
4,5,01/22/2020,Gansu,Mainland China,1/22/2020 17:00,0.0,0.0,0.0


In [2]:
df.tail()

,SNo,ObservationDate,Province/State,Country/Region,Last Update,Confirmed,Deaths,Recovered
6717,6718,03/18/2020,NaN,Guernsey,2020-03-17T18:33:03,0.0,0.0,0.0
6718,6719,03/18/2020,NaN,Jersey,2020-03-17T18:33:03,0.0,0.0,0.0
6719,6720,03/18/2020,NaN,Puerto Rico,2020-03-17T16:13:14,0.0,0.0,0.0
6720,6721,03/18/2020,NaN,Republic of the Congo,2020-03-17T21:33:03,0.0,0.0,0.0
6721,6722,03/18/2020,NaN,The Gambia,2020-03-18T14:13:56,0.0,0.0,0.0


In [5]:
non_zero_deathts=df[df['Deaths']>0]
non_zero_deathts.head()

,SNo,ObservationDate,Province/State,Country/Region,Last Update,Confirmed,Deaths,Recovered
13,14,01/22/2020,Hubei,Mainland China,1/22/2020 17:00,444.0,17.0,28.0
47,48,01/23/2020,Hebei,Mainland China,1/23/20 17:00,1.0,1.0,0.0
51,52,01/23/2020,Hubei,Mainland China,1/23/20 17:00,444.0,17.0,28.0
84,85,01/24/2020,Hubei,Mainland China,1/24/20 17:00,549.0,24.0,31.0
103,104,01/24/2020,Heilongjiang,Mainland China,1/24/20 17:00,4.0,1.0,0.0


In [6]:
non_zero_deathts.tail()

,SNo,ObservationDate,Province/State,Country/Region,Last Update,Confirmed,Deaths,Recovered
6645,6646,03/18/2020,NaN,Cuba,2020-03-18T16:59:35,7.0,1.0,0.0
6647,6648,03/18/2020,NaN,Guyana,2020-03-17T10:53:03,7.0,1.0,0.0
6652,6653,03/18/2020,NaN,Guatemala,2020-03-17T12:13:16,6.0,1.0,0.0
6681,6682,03/18/2020,NaN,Sudan,2020-03-18T12:13:09,2.0,1.0,0.0
6709,6710,03/18/2020,Cayman Islands,UK,2020-03-16T14:53:04,1.0,1.0,0.0


In [8]:
df['ObservationDate']=pd.to_datetime(df['ObservationDate']) 
df['Infected']=df['Confirmed']-df['Recovered']-df['Deaths'] 
#aggregate daily global totals
daily_totals=df.groupby('ObservationDate')[['Confirmed', 'Infected', 'Recovered', 'Deaths']].sum().reset_index() 
daily_totals=daily_totals.sort_values('ObservationDate').reset_index(drop=True) 
daily_totals['Day']=(daily_totals['ObservationDate']-daily_totals['ObservationDate'].min()).dt.days 
daily_totals.head()

,ObservationDate,Confirmed,Infected,Recovered,Deaths,Day
0,2020-01-22,555.0,510.0,28.0,17.0,0
1,2020-01-23,653.0,605.0,30.0,18.0,1
2,2020-01-24,941.0,879.0,36.0,26.0,2
3,2020-01-25,1438.0,1357.0,39.0,42.0,3
4,2020-01-26,2118.0,2010.0,52.0,56.0,4


In [13]:
#model training and 7-day forecasting
x=daily_totals[['Day']] 
targets=['Confirmed', 'Infected', 'Recovered', 'Deaths'] 

#future time range (next 7 days)
last_day=daily_totals['Day'].max() 

future_day_values=[last_day+i for i in range(1,8)] 
future_days=pd.DataFrame({'Day': future_day_values})

last_date=daily_totals['ObservationDate'].max() 
future_dates=[last_date+pd.Timedelta(days=i) for i in range(1,8)] 

#fit linear regression per target and project future values 
predictions={'ObservationDate': [d.strftime('%m/%d/%Y') for d in future_dates]} 

for target in targets: 
    y=daily_totals[target] 

    model= LinearRegression() 
    model.fit(x,y) 

    pred=model.predict(future_days) 
    predictions[target]=np.maximum(0,np.round(pred)).astype(int) 

forecast_df=pd.DataFrame(predictions) 
forecast_df

,ObservationDate,Confirmed,Infected,Recovered,Deaths
0,03/19/2020,158934,80257,72904,5773
1,03/20/2020,161970,81581,74498,5892
2,03/21/2020,165006,82904,76091,6010
3,03/22/2020,168041,84228,77685,6128
4,03/23/2020,171077,85552,79278,6247
5,03/24/2020,174113,86876,80872,6365
6,03/25/2020,177149,88200,82466,6483


In [18]:
df.nlargest(7,'Deaths')

,SNo,ObservationDate,Province/State,Country/Region,Last Update,Confirmed,Deaths,Recovered,Infected
6438,6439,2020-03-18,Hubei,Mainland China,2020-03-18T12:13:09,67800.0,3122.0,56927.0,7751.0
6162,6163,2020-03-17,Hubei,Mainland China,2020-03-17T11:53:10,67799.0,3111.0,56003.0,8685.0
5890,5891,2020-03-16,Hubei,Mainland China,2020-03-16T14:38:45,67798.0,3099.0,55142.0,9557.0
5632,5633,2020-03-15,Hubei,Mainland China,2020-03-15T18:20:18,67794.0,3085.0,54288.0,10421.0
5383,5384,2020-03-14,Hubei,Mainland China,2020-03-14T10:13:09,67790.0,3075.0,52960.0,11755.0
5153,5154,2020-03-13,Hubei,Mainland China,2020-03-13T11:09:03,67786.0,3062.0,51553.0,13171.0
4935,4936,2020-03-12,Hubei,Mainland China,2020-03-12T09:53:06,67781.0,3056.0,50318.0,14407.0
